# Prepare pinyin acoustic embeddings

Build one immutable acoustic bundle from the complete pinyin sound inventory. The bundle contains portable pinyin vectors and their audited inventory; the cache only avoids repeated computation and is not an output dataset.


## Project imports

Locate the local project from this notebook or any nested working directory, then import the shared preparation SDK from `src`.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'datasets').is_dir() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Could not find a project root containing datasets/ and src/')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from phonetics import prepare_acoustic_bundle


## Configuration

The sound manifest pins the full inventory. Set exact model and processor revisions before enabling inference. `layer=-1` selects the last hidden layer, `mean_all` pools every output frame, and `padding="none"` preserves the original waveform duration.


In [ ]:
RUN_INFERENCE = False
SOUNDS_DIR = PROJECT_ROOT / 'datasets/source-data/sounds'
SOUND_MANIFEST_PATH = PROJECT_ROOT / 'datasets/source-data/sounds-manifest/chinese-sounds-v1/sounds_manifest.csv'
SOUND_MANIFEST_SHA256 = 'a5c4b7ac719804cc3cfdb853ff9365d673d1a2ca98c8e37d0ea275020533acaa'
OUTPUT_ROOT = PROJECT_ROOT / 'datasets/pinyin-embeddings'
CACHE_ROOT = PROJECT_ROOT / '.cache/acoustic-embeddings'

CONFIG = {
    'bundle_id': 'duyu-chinese-lessons-zh-cn-v1',
    'output_root': str(OUTPUT_ROOT),
    'cache_root': str(CACHE_ROOT),
    'sounds': {'path': str(SOUNDS_DIR), 'manifest': str(SOUND_MANIFEST_PATH), 'sha256': SOUND_MANIFEST_SHA256, 'tone_convention': 'explicit'},
    'model': {'id': 'REPLACE_WITH_MODEL_ID', 'revision': 'REPLACE_WITH_EXACT_REVISION', 'processor': 'REPLACE_WITH_PROCESSOR_ID', 'layer': -1, 'pooling': 'mean_all', 'sample_rate': 16000, 'mono': True, 'padding': 'none', 'dtype': 'float32', 'device': 'cpu'},
}


## Build the bundle

Enable the guarded cell only after reviewing local paths and replacing every model placeholder. Existing output bundles are never overwritten.


In [ ]:
if RUN_INFERENCE:
    assert 'REPLACE_WITH' not in repr(CONFIG)
    bundle_path = prepare_acoustic_bundle(CONFIG)
    print(bundle_path)
else:
    print('Inference disabled; review local paths, exact revision, and device first.')
